# Stage 7: Reranking

After fusion, a cross-encoder reads the question together with each of the top fused candidates and scores how
well it answers it; the best few are kept. It is switchable (`RERANK=on|off`, or `search(..., rerank=False)`).
Code: `Reranker` and `Retriever.search` in `src/handbook_qa/retrieval.py`.

This notebook compares three local cross-encoders that fastembed runs on ONNX, then shows the switch. Models are
downloaded on first use into `data/models/`.

The spot-check questions are a sanity check, **not an evaluation**: with 11 questions, differences of one or two
ranks are noise. They show whether a reranker breaks things and roughly what it costs in time.

In [1]:
import gc
import json
import time
from dataclasses import replace

import pandas as pd
import psutil

from handbook_qa.config import load_settings
from handbook_qa.retrieval import Reranker, Retriever

settings = load_settings()
r = settings.retrieval
print(f"rerank default: {r.rerank} | model: {r.rerank_model} | candidates: {r.rerank_candidates} | keep: {r.rerank_top_n}")

# The first stage (dense + BM25 + RRF) is shared; rerankers are swapped in below.
retriever = Retriever.from_settings(replace(settings, retrieval=replace(r, rerank=False)))
retriever.rerank_candidates, retriever.rerank_top_n = r.rerank_candidates, r.rerank_top_n
SPOT = json.loads(open("spot_check_questions.json", encoding="utf-8").read())["questions"]
KNOWN = [c for c in SPOT if c["expected"]]
pd.set_option("display.max_colwidth", 80)


def first_rank(ids, expected):
    for n, item_id in enumerate(ids, start=1):
        if any(item_id == e or item_id.startswith(e) for e in expected):
            return n
    return None

rerank default: True | model: Xenova/ms-marco-MiniLM-L-6-v2 | candidates: 30 | keep: 5


C:\Users\shaba\Documents\extra project\paradigm_assessment\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Candidates

| model | size | max input | note |
|---|---|---|---|
| Xenova/ms-marco-MiniLM-L-6-v2 | 0.08 GB | 512 tokens | fastest; reads only the first ~512 tokens of a chunk (chunks run to ~900) |
| jinaai/jina-reranker-v1-turbo-en | 0.15 GB | 8K tokens | small, reads whole chunks |
| BAAI/bge-reranker-v2-m3-int8 | 0.57 GB | 8K tokens | larger and generally stronger; int8-quantised |

In [2]:
MODELS = ["Xenova/ms-marco-MiniLM-L-6-v2", "jinaai/jina-reranker-v1-turbo-en", "BAAI/bge-reranker-v2-m3-int8"]
process = psutil.Process()

baseline = {c["question"]: [h.item.id for h in retriever.search(c["question"], k=r.rerank_top_n, rerank=False)]
            for c in SPOT}
results, timing = {"fused (no rerank)": baseline}, []
for name in MODELS:
    started = time.perf_counter()
    retriever.reranker = Reranker(name, settings.embed.model_cache, settings.embed.threads)
    load = time.perf_counter() - started
    per_question, ranked = [], {}
    for case in SPOT:
        started = time.perf_counter()
        ranked[case["question"]] = [h.item.id for h in retriever.search(case["question"], rerank=True)]
        per_question.append(time.perf_counter() - started)
    results[name] = ranked
    timing.append({"model": name, "load s": round(load, 1),
                   "median s per question": round(sorted(per_question)[len(per_question) // 2], 2),
                   "max s": round(max(per_question), 2), "process GB": round(process.memory_info().rss / 1e9, 2)})
    retriever.reranker = None
    gc.collect()
pd.DataFrame(timing)

,model,load s,median s per question,max s,process GB
0,Xenova/ms-marco-MiniLM-L-6-v2,0.3,3.22,3.37,1.33
1,jinaai/jina-reranker-v1-turbo-en,0.4,4.40,4.48,1.54
2,BAAI/bge-reranker-v2-m3-int8,2.2,48.37,51.19,3.15


## Where the expected answer lands in the kept top 5

Blank = not in the top 5.

In [3]:
ranks = pd.DataFrame({name: [first_rank(ranked[c["question"]], c["expected"]) for c in KNOWN]
                      for name, ranked in results.items()},
                     index=[c["question"] for c in KNOWN])
ranks

,fused (no rerank),Xenova/ms-marco-MiniLM-L-6-v2,jinaai/jina-reranker-v1-turbo-en,BAAI/bge-reranker-v2-m3-int8
Who should I ask for new artwork or merch?,1,1,1,1
"If I start on July 21st, how is my commission calculated during ramp?",2,2,1,1
Which communication channels are best and worst at PostHog?,1,1,4,1
How much credit do Y Combinator companies get?,1,1,1,1
How do I sign my git commits?,1,1,2,1
How do I raise an incident?,3,2,1,2
How much RAM does the 4-core codespace have?,1,1,1,1
How do we handle refund requests?,1,1,1,1
What are the stages of the engineering hiring process?,1,1,1,1
What is the policy for spending company money?,1,1,1,1


In [4]:
pd.DataFrame({name: {"in top 5": int(col.notna().sum()), "rank 1": int((col == 1).sum()),
                     "mean rank when found": round(col.dropna().mean(), 2)}
              for name, col in ranks.items()}).T

,in top 5,rank 1,mean rank when found
fused (no rerank),11.0,9.0,1.27
Xenova/ms-marco-MiniLM-L-6-v2,11.0,9.0,1.18
jinaai/jina-reranker-v1-turbo-en,11.0,8.0,1.45
BAAI/bge-reranker-v2-m3-int8,11.0,10.0,1.09


Note on images: each image's description is in its own record *and* in the chunk where it sits. Cross-encoders
tend to prefer the chunk (more surrounding text) over the image record. The chunk still carries the answer and the
image ID, so both count as found here; the first version of this check counted only the image record and wrongly
showed the 8K rerankers losing the logo answer.

## Top 5 for each question, per reranker

In [5]:
pd.DataFrame({name: [", ".join(ranked[c["question"]][:3]) for c in SPOT] for name, ranked in results.items()},
             index=[c["question"] for c in SPOT])

,fused (no rerank),Xenova/ms-marco-MiniLM-L-6-v2,jinaai/jina-reranker-v1-turbo-en,BAAI/bge-reranker-v2-m3-int8
Who should I ask for new artwork or merch?,"brand/art-requests#01, company/merch-store#01, company/merch-store#02","brand/art-requests#01, marketing/customer-case-studies#02, marketing/ownersh...","brand/art-requests#01, marketing/ownership#01, product/visiting-customers#02","brand/art-requests#01, marketing/ownership#01, marketing/customer-case-studi..."
"If I start on July 21st, how is my commission calculated during ramp?","growth/sales/how-we-work#05, img-p0617-x2727, growth/sales/how-we-work#07","growth/sales/how-we-work#05, growth/sales/new-hire-onboarding#06, growth/sal...","growth/sales/new-hire-onboarding#06, growth/sales/how-we-work#09, growth/sal...","growth/sales/new-hire-onboarding#06, growth/sales/how-we-work#09, growth/sal..."
Which communication channels are best and worst at PostHog?,"img-p0041-x258, company/communication#06, company/communication#11","img-p0041-x258, company/communication#06, cs-and-onboarding/customer-success#02","company/communication#06, growth/sales/communications-templates-incidents#01...","img-p0041-x258, growth/sales/communications-templates-incidents#01, company/..."
How much credit do Y Combinator companies get?,"brand/startups#01, brand/startups#03, brand/startups#02","brand/startups#01, brand/startups#03, brand/startups#02","brand/startups#03, brand/startups#02, brand/startups#01","brand/startups#01, brand/startups#02, brand/startups#03"
How do I sign my git commits?,"engineering/security#02, engineering/posthog-com/developing-the-website#13, ...","engineering/security#02, engineering/security#03, img-p0401-x1908","engineering/security#03, engineering/security#02, engineering/posthog-com/de...","engineering/security#02, engineering/security#01, engineering/security#03"
How do I raise an incident?,"support/support-incident-response#01, support/support-incident-response#02, ...","support/support-incident-response#01, engineering/operations/incidents#02, e...","engineering/operations/incidents#03, engineering/operations/incidents#02, su...","support/support-incident-response#01, engineering/operations/incidents#01, e..."
How much RAM does the 4-core codespace have?,"img-p0355-x1727, img-p0355-x1726, img-p0356-x1730","img-p0355-x1727, engineering/posthog-com/developing-the-website#01, people/s...","img-p0355-x1727, engineering/posthog-com/developing-the-website#01, engineer...","img-p0355-x1727, engineering/posthog-com/developing-the-website#01, people/s..."
How do we handle refund requests?,"growth/sales/refunds#02, growth/sales/refunds#01, growth/sales/refunds#04","growth/sales/refunds#01, growth/sales/refunds#02, growth/sales/refunds#05","growth/sales/refunds#01, growth/sales/refunds#02, growth/sales/refunds#05","growth/sales/refunds#01, growth/sales/refunds#02, growth/sales/refunds#05"
What are the stages of the engineering hiring process?,"people/hiring-process/engineering-hiring#03, people/hiring-process/engineeri...","people/hiring-process/engineering-hiring#03, people/hiring-process/engineeri...","people/hiring-process/engineering-hiring#03, people/hiring-process/engineeri...","people/hiring-process/engineering-hiring#03, people/hiring-process/engineeri..."
What is the policy for spending company money?,"people/spending-money#01, people/spending-money#03, people/spending-money#02","people/spending-money#01, people/spending-money#02, people/spending-money#03","people/spending-money#01, people/spending-money#02, people/spending-money#05","people/spending-money#01, company/small-teams#06, people/spending-money#02"


## The switch

The configured model (`RERANK_MODEL`) loads when reranking is on. Per call, `rerank=False` skips it and returns the
fused top 8; with it on, the top 30 fused candidates are rescored and the best 5 kept.

In [6]:
retriever.reranker = Reranker(r.rerank_model, settings.embed.model_cache, settings.embed.threads)
question = "What should the logo never look like?"
for flag in (False, True):
    hits = retriever.search(question, rerank=flag)
    print(f"\nrerank={flag}: {len(hits)} results")
    for n, h in enumerate(hits, 1):
        score = f"rerank {h.rerank_score:6.3f}" if h.rerank_score is not None else ""
        print(f"  {n}. {h.item.id:45} fused #{h.fused_rank:<3} {score}")


rerank=False: 8 results
  1. company/brand-assets#02                       fused #1   
  2. img-p0038-x244                                fused #2   
  3. company/brand-assets#01                       fused #3   
  4. img-p0037-x212                                fused #4   
  5. brand/style-guide#02                          fused #5   
  6. company/brand-assets#07                       fused #6   
  7. img-p0358-x1741                               fused #7   
  8. img-p0043-x266                                fused #8   



rerank=True: 5 results
  1. company/brand-assets#02                       fused #1   rerank -1.821
  2. marketing/customer-case-studies#01            fused #16  rerank -2.900
  3. img-p0038-x244                                fused #2   rerank -3.056
  4. company/brand-assets#01                       fused #3   rerank -3.898
  5. brand/style-guide#02                          fused #5   rerank -6.688
